In [1]:
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Dict, Optional, Tuple
from itertools import product
import pandas as pd
import pyodbc

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from data_fetcher import RecOpsData

In [4]:
data_fetcher = RecOpsData()
data_fetcher.initialize_state()

C:\Users\SW6720\Github\ec_ops\data_fetcher.py:139: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  self.pos_df = pd.read_sql(self.pos_query, conn)
C:\Users\SW6720\Github\ec_ops\data_fetcher.py:140: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  self.pnl_df = pd.read_sql(self.pnl_query, conn)
C:\Users\SW6720\Github\ec_ops\data_fetcher.py:241: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  self.pos_level_pnl = pd.read_sql(query, conn)


In [5]:
delivs_sql_path = "./sql/deliv_excel.sql"
with open(delivs_sql_path, "r", encoding="utf-8") as f:
            delivs_sql = f.read()

In [7]:
delivs_df = data_fetcher.execute_znet_query(delivs_sql)

C:\Users\SW6720\Github\ec_ops\data_fetcher.py:121: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [8]:
delivs_df.head()

,REPORT_DATE,DMO,TRADE_TYPE2,TNUM_XREF,TNUMGROUP,GROUP_DSC2,TRADER_NAME,BOOK,TRADE_BS,ZKEY,...,STRIKE,TRADE_PRICE,MKT,YEAR,LOCATION_DESCR,STATFULL,DEL_DATEEND,AMOUNT,4STAMP,MEMO
0,2026-03-25,2028-02-01,ALLOWANCE PHYSICAL,A63637,OQP532,DEFAULT,Steve Adams,RPWR,B,104065960.0,...,0.0,0.6,USWIBH,2027,US Green-e Wind Back Half REC 2027,BC2,2028-02-29,34893.0,None,Delivery Date revised per amended and restated...
1,2026-03-25,2028-02-01,ALLOWANCE PHYSICAL,A63637,OQP532,DEFAULT,Steve Adams,ROWN,S,104065961.0,...,0.0,0.6,USWIBH,2027,US Green-e Wind Back Half REC 2027,BC2,2028-02-29,-34893.0,None,Delivery Date revised per amended and restated...
2,2026-03-25,2026-02-01,ALLOWANCE PHYSICAL,A63637,OQP545,DEFAULT,Steve Adams,RPWR,B,104066002.0,...,0.0,0.6,USWIFH,2025,US Green-e Wind Front Half REC 2025,BC2,2026-02-27,33982.0,4,Delivery Date revised per amended and restated...
3,2026-03-25,2026-02-01,ALLOWANCE PHYSICAL,A63637,OQP545,DEFAULT,Steve Adams,ROWN,S,104066003.0,...,0.0,0.6,USWIFH,2025,US Green-e Wind Front Half REC 2025,BC2,2026-02-27,-33982.0,4,Delivery Date revised per amended and restated...
4,2026-03-25,2026-02-01,ALLOWANCE PHYSICAL,A63637,OQP546,DEFAULT,Steve Adams,RPWR,B,104066004.0,...,0.0,0.6,USWIFH,2025,US Green-e Wind Front Half REC 2025,BC2,2026-02-27,35232.0,4,Delivery Date revised per amended and restated...


In [13]:
query = """SELECT TOP (1000) [ID]
      ,[Market]
      ,[Location]
      ,[Segment]
      ,[ObservationDate]
      ,[EffectiveDate]
      ,[CurveValue]
      ,[CreatedDate]
  FROM [DataWarehouse].[TraderMarks].[CurveValue]
  WHERE [Market] = 'DCT1'
  AND [Location] = 2027"""

In [135]:
curves = {}

In [87]:
products = ['TXSABH','PJMTRI']
vints = [2027]
for prod, vint in product(products,vints):
    prod_str = f"{prod}_{vint}"
    query = f"""SELECT TOP (1000) [ID]
          ,[Market]
          ,[Location]
          ,[Segment]
          ,[ObservationDate]
          ,[EffectiveDate]
          ,[CurveValue]
          ,[CreatedDate]
      FROM [DataWarehouse].[TraderMarks].[CurveValue]
      WHERE [Market] = '{prod}'
      AND [Location] = {vint}"""
    curves[prod_str] = data_fetcher.execute_mssql_query(query)
    #curves[prod_str]['CreatedDate'] = curves[prod_str]['CreatedDate'].apply(lambda x: x.normalize())
    curves[prod_str] = curves[prod_str].rename(columns = {"CurveValue":prod_str})

<string>:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


In [114]:
prods = list(curves.keys())

In [116]:
merge_cols = [[prod,'ObservationDate'] for prod in prods]
#curves['DCT1_2026'][]
i = 0
j = 1
data = curves[prods[i]][merge_cols[i]].merge(curves[prods[j]][merge_cols[j]], on = 'ObservationDate', how = 'outer').set_index('ObservationDate')

data['spread'] = data[prods[1]]-data[prods[0]]
data.head()

KeyError: 'TXWIFH_2024'

In [85]:
data.to_clipboard()

In [118]:
products = ['TXWIFH']
vints = [2027,2028, 2030, 2032]
for prod, vint in product(products,vints):
    prod_str = f"{prod}_{vint}"
    query = f"""SELECT TOP (1000) [ID]
          ,[Market]
          ,[Location]
          ,[Segment]
          ,[ObservationDate]
          ,[EffectiveDate]
          ,[CurveValue]
          ,[CreatedDate]
      FROM [DataWarehouse].[TraderMarks].[CurveValue]
      WHERE [Market] = '{prod}'
      AND [Location] = {vint}"""
    curves[prod_str] = data_fetcher.execute_mssql_query(query)
    #curves[prod_str]['CreatedDate'] = curves[prod_str]['CreatedDate'].apply(lambda x: x.normalize())
    curves[prod_str] = curves[prod_str].rename(columns = {"CurveValue":prod_str})
prods = list(curves.keys())

<string>:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


In [136]:
products = ['PAT1','DCT1','PJMTRI']
vints = [2027]
for prod, vint in product(products,vints):
    prod_str = f"{prod}_{vint}"
    query = f"""SELECT TOP (1000) [ID]
          ,[Market]
          ,[Location]
          ,[Segment]
          ,[ObservationDate]
          ,[EffectiveDate]
          ,[CurveValue]
          ,[CreatedDate]
      FROM [DataWarehouse].[TraderMarks].[CurveValue]
      WHERE [Market] = '{prod}'
      AND [Location] = {vint}"""
    curves[prod_str] = data_fetcher.execute_mssql_query(query)
    #curves[prod_str]['CreatedDate'] = curves[prod_str]['CreatedDate'].apply(lambda x: x.normalize())
    curves[prod_str] = curves[prod_str].rename(columns = {"CurveValue":prod_str})
prods = list(curves.keys())

<string>:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


In [137]:
merge_cols = [[prod,'ObservationDate'] for prod in prods]
#curves['DCT1_2026'][]
i = 0
j = 1
data = curves[prods[i]][merge_cols[i]].merge(curves[prods[j]][merge_cols[j]], on = 'ObservationDate', how = 'outer')

In [139]:
data.to_clipboard()

In [138]:
for j in range(2,len(prods)):
    data = data.merge(curves[prods[j]][merge_cols[j]], on = 'ObservationDate', how = 'outer')

In [56]:
#use me for two product data sets

merge_cols = [[prod,'ObservationDate'] for prod in prods]
#curves['DCT1_2026'][]
i = 0
j = 1
data = curves[prods[i]][merge_cols[i]].merge(curves[prods[j]][merge_cols[j]], on = 'ObservationDate', how = 'outer').set_index('ObservationDate')

data['spread'] = data[prods[1]]-data[prods[0]]
data.head()

,TXWIBH_2026,TXWIBH_2030,spread
ObservationDate,,,
2023-06-27,4.1,3.7,-0.4
2023-06-28,3.8,3.7,-0.1
2023-06-29,3.8,3.7,-0.1
2023-06-30,3.8,3.7,-0.1
2023-07-05,3.8,3.7,-0.1


In [57]:
filt = data['spread'].notna()
data[filt].to_clipboard()

In [15]:
query = """SELECT TOP (1000) [ID]
      ,[Market]
      ,[Location]
      ,[Segment]
      ,[ObservationDate]
      ,[EffectiveDate]
      ,[CurveValue]
      ,[CreatedDate]
  FROM [DataWarehouse].[TraderMarks].[CurveValue]
  WHERE [Market] = 'PJMTRI'
  AND [Location] = 2027"""
tri = data_fetcher.execute_mssql_query(query)
tri.head()

<string>:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


,ID,Market,Location,Segment,ObservationDate,EffectiveDate,CurveValue,CreatedDate
0,2317308,PJMTRI,2027,RTC,2021-08-27,2027-01-01,14.75,2021-08-27 13:45:46.810
1,2317826,PJMTRI,2027,RTC,2021-08-30,2027-01-01,14.75,2021-08-30 14:16:00.227
2,2318293,PJMTRI,2027,RTC,2021-08-31,2027-01-01,14.75,2021-08-31 14:01:54.220
3,2319032,PJMTRI,2027,RTC,2021-09-01,2027-01-01,14.75,2021-09-01 14:56:26.193
4,2319227,PJMTRI,2027,RTC,2021-09-02,2027-01-01,14.75,2021-09-02 14:06:37.407


In [16]:
tri['CreatedDate'] = tri['CreatedDate'].apply(lambda x: x.normalize())
dc_c1['CreatedDate'] = tri['CreatedDate'].apply(lambda x: x.normalize())

0     2021-08-27
1     2021-08-30
2     2021-08-31
3     2021-09-01
4     2021-09-02
         ...    
351   2026-03-24
352   2026-03-25
353   2026-03-26
354   2026-03-27
355   2026-03-30
Name: CreatedDate, Length: 356, dtype: datetime64[ns]